## Trace Analytics

Read-only static analysis for agent traces in `__data1/agent-traces.json` with top-level `runs`.  Each trace step is one backend decision based on `candidates` selection.

Research focus:
- safety-vs-progress candidate generation, heuristic scoring, and model selection.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd

# If you open the notebook from another working directory, set this manually.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "__data1").exists() and (REPO_ROOT.parent / "__data1").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "__data1"
TRACES_PATH = DATA_DIR / "agent-traces.json"

def load_json(path: Path, default: dict[str, Any]) -> dict[str, Any]:
    if not path.exists():
        print(f"missing: {path}")
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"invalid JSON in {path}: {exc}") from exc

trace_store = load_json(TRACES_PATH, {"version": 3, "runs": {}})
runs = trace_store.get("runs") or {}

print(f"data dir:  {DATA_DIR}")
print(f"runs:      {len(runs)}")


### Normalize JSON To Tables

The dataframe builders read the current V3 trace schema. `steps_df` has one row per decision step; `candidates_df` has one row per model-visible candidate.

In [ ]:
VALID_CANDIDATE_LANES = {"safety", "progress", "environment"}

def iso_to_datetime(value: Any) -> pd.Timestamp:
    if not value:
        return pd.Timestamp("NaT")
    return pd.to_datetime(value, utc=True, errors="coerce")


def list_len(value: Any) -> int:
    return len(value) if isinstance(value, list) else 0


def short_trace_id(value: Any) -> str:
    return str(value)[:8]


short_trace_ids = [short_trace_id(trace_id) for trace_id in runs]
if len(short_trace_ids) != len(set(short_trace_ids)):
    raise ValueError("traceId first-eight-character collision")


def candidate_lane(candidate: dict[str, Any]) -> str:
    lane = candidate.get("lane")
    return lane if lane in VALID_CANDIDATE_LANES else "other"


def build_steps_df(runs: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for trace_id, run in runs.items():
        for index, step in enumerate(run.get("steps") or []):
            state = step.get("state") or {}
            runner = state.get("runner") or {}
            gold = state.get("gold") or {}
            guard_risk = state.get("guardRisk") or {}
            validation = step.get("validation") or {}
            loop = step.get("loopMonitor") or {}
            suppressed = loop.get("suppressedCandidates") or []
            action = step.get("action") or {}
            rows.append({
                "traceId": short_trace_id(trace_id),
                "stepIndex": step.get("stepIndex", index),
                "createdAt": iso_to_datetime(step.get("createdAt")),
                "tick": state.get("tick"),
                "gameState": state.get("gameState"),
                "godMode": state.get("godMode"),
                "runnerX": runner.get("x"),
                "runnerY": runner.get("y"),
                "runnerAction": runner.get("action"),
                "goldComplete": gold.get("complete"),
                "goldRemaining": gold.get("remainingCount"),
                "visibleGoldCount": list_len(gold.get("visiblePositions")),
                "guardRisk": guard_risk.get("risk"),
                "requestedCandidateId": validation.get("requestedCandidateId"),
                "selectedCandidateId": step.get("selectedCandidateId"),
                "selectedCandidateKind": step.get("selectedCandidateKind"),
                "keyCode": action.get("keyCode"),
                "ticks": action.get("ticks"),
                "validationFallbackUsed": validation.get("fallbackUsed"),
                "validationFallbackReason": validation.get("fallbackReason"),
                "loopActive": loop.get("active"),
                "loopType": loop.get("type"),
                "suppressedCandidateCount": len(suppressed),
                "candidateCount": list_len(step.get("candidates")),
            })
    return pd.DataFrame(rows)


def build_candidates_df(runs: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for trace_id, run in runs.items():
        for index, step in enumerate(run.get("steps") or []):
            validation = step.get("validation") or {}
            requested_id = validation.get("requestedCandidateId")
            selected_id = step.get("selectedCandidateId")
            for rank, candidate in enumerate(step.get("candidates") or [], start=1):
                first_action = candidate.get("firstAction") or {}
                candidate_id = candidate.get("id")
                kind = candidate.get("kind")
                rows.append({
                    "traceId": short_trace_id(trace_id),
                    "stepIndex": step.get("stepIndex", index),
                    "rank": rank,
                    "candidateId": candidate_id,
                    "kind": kind,
                    "lane": candidate_lane(candidate),
                    "score": candidate.get("score"),
                    "requested": candidate_id == requested_id,
                    "selected": candidate_id == selected_id,
                    "keyCode": first_action.get("keyCode"),
                    "ticks": first_action.get("ticks"),
                    "reason": first_action.get("reason"),
                    "target": candidate.get("target"),
                })
    return pd.DataFrame(rows)

steps_df = build_steps_df(runs)
candidates_df = build_candidates_df(runs)

for name, df in [
    ("steps_df", steps_df),
    ("candidates_df", candidates_df),
]:
    print(f"{name}: {df.shape}")


### Safety vs Progress Candidate Analysis

Candidate kinds are grouped into three lanes:

- **safety**: avoid, trap, retreat from, or wait out guard danger;
- **progress**: collect gold, traverse routes, or reach the exit;
- **environment**: wait for a state change or recheck when no other candidate is available.

The summary counts which decision steps expose safety versus progress candidates, or both. Table 1 shows lane availability. Table 2 classifies singleton steps. A **singleton** is a step with exactly one model-visible candidate; other proposals may still appear in `candidateAudit` as rejected or suppressed. In Tables 2 and 3, **forced safety** means a non-wait, non-emergency safety-lane singleton.

In [ ]:
KEYS = ["traceId", "stepIndex"]

if steps_df.empty or candidates_df.empty:
    print("No trace decisions available.")
else:
    lane_counts = candidates_df.pivot_table(
        index=KEYS, columns="lane", values="candidateId", aggfunc="count", fill_value=0
    )
    for lane in ["safety", "progress", "environment", "other"]:
        if lane not in lane_counts.columns:
            lane_counts[lane] = 0

    decisions_df = (
        steps_df[KEYS].set_index(KEYS)
        .join(lane_counts, how="left")
    )
    lane_count_columns = ["safety", "progress", "environment", "other"]
    decisions_df[lane_count_columns] = decisions_df[lane_count_columns].fillna(0).astype(int)
    decisions_df["hasSafety"] = decisions_df["safety"] > 0
    decisions_df["hasProgress"] = decisions_df["progress"] > 0

    def availability_label(row: pd.Series) -> str:
        if row["hasSafety"] and row["hasProgress"]:
            return "both"
        if row["hasSafety"]:
            return "safety only"
        if row["hasProgress"]:
            return "progress only"
        return "neither"

    decisions_df["availability"] = decisions_df.apply(availability_label, axis=1)
    availability_order = ["safety only", "progress only", "both", "neither"]
    availability_summary = (
        decisions_df["availability"].value_counts()
        .reindex(availability_order, fill_value=0)
        .rename_axis("candidate lane")
        .rename("steps")
        .sort_values(ascending=False)
    )
    availability_summary.loc["total"] = availability_summary.sum()
    print("Table 1: Candidate lane totals")
    display(availability_summary.to_frame())

    singleton_candidates = candidates_df[
        (candidates_df.groupby(KEYS)["candidateId"].transform("size") == 1)
        & candidates_df["lane"].ne("other")
    ].copy()
    wait_kinds = {
        "wait_and_recheck", "wait_for_dig_completion", "wait_for_floor_refill",
        "wait_for_guard_clearance", "wait_for_trap_resolution",
    }

    def singleton_label(row: pd.Series) -> str:
        if row["kind"] == "emergency_hold":
            return "emergency hold"
        if row["kind"] in wait_kinds:
            return "environment"
        if row["lane"] == "safety":
            return "forced safety"
        if row["lane"] == "progress":
            return "forced progress"
        raise ValueError(
            f"unclassified singleton: {row['candidateId']} ({row['lane']})"
        )

    singleton_order = [
        "emergency hold", "forced safety", "forced progress", "environment"
    ]
    singleton_candidates["singletonType"] = singleton_candidates.apply(
        singleton_label, axis=1
    )
    singleton_summary = (
        singleton_candidates["singletonType"].value_counts()
        .reindex(singleton_order, fill_value=0)
        .rename_axis("singleton type")
        .rename("steps")
        .sort_values(ascending=False)
    )
    singleton_summary.loc["total"] = singleton_summary.sum()
    print("Table 2: Singleton type totals")
    display(singleton_summary.to_frame())

    availability_by_run = (
        decisions_df.groupby(["traceId", "availability"]).size().unstack(fill_value=0)
        .reindex(
            columns=["safety only", "progress only", "both"],
            fill_value=0,
        )
    )
    singleton_by_run = (
        singleton_candidates.groupby(["traceId", "singletonType"])
        .size().unstack(fill_value=0)
        .reindex(columns=singleton_order, fill_value=0)
    )
    singleton_by_run = singleton_by_run[["forced safety", "forced progress"]]
    singleton_count_by_run = singleton_candidates.groupby("traceId").size()
    safety_rejection_by_run = pd.Series({
        short_trace_id(trace_id): sum(
            1
            for step in (run.get("steps") or [])
            for audit in (step.get("candidateAudit") or [])
            if audit.get("disposition") == "safety_rejection"
        )
        for trace_id, run in runs.items()
    })
    newest_run_order = (
        pd.Series({
            short_trace_id(trace_id): iso_to_datetime(run.get("updatedAt"))
            for trace_id, run in runs.items()
        })
        .sort_values(ascending=False, na_position="last")
        .index
    )
    per_run_summary = pd.concat(
        {"candidate lane": availability_by_run,
         "singleton type": pd.concat(
             {"# singleton": singleton_count_by_run, **{column: singleton_by_run[column] for column in singleton_by_run}},
             axis=1,
         )},
        axis=1,
    ).fillna(0).astype(int).reindex(newest_run_order)
    per_run_summary.insert(
        0,
        "traceId",
        per_run_summary.index,
    )
    per_run_summary.insert(
        1,
        "# steps",
        decisions_df.groupby(level="traceId").size().reindex(newest_run_order).to_numpy(),
    )
    per_run_summary.insert(
        3,
        "safety rejection",
        safety_rejection_by_run.reindex(newest_run_order).to_numpy(),
    )
    per_run_summary.columns = [
        "traceId", "# steps", "safety lane", "safety rejection",
        "progress lane", "both lanes",
        "# singleton", "forced safety", "forced progress",
    ]
    per_run_summary = per_run_summary.reset_index(drop=True)
    per_run_summary.loc[len(per_run_summary)] = {
        "traceId": "total",
        **{
            column: per_run_summary[column].sum()
            for column in per_run_summary.columns
            if column != "traceId"
        },
    }
    print("Table 3: Per-run summary")
    display(per_run_summary)
